In [1]:
import numpy as np
import tensorflow as tf 

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Embedding,Dense

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
text = """Shall I compare thee to a summer's day? Thou art more lovely and more temperate.
Rough winds do shake the darling buds of May, And summer's lease hath all too short a date."""

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

In [4]:
print(tokenizer.word_index)

{'a': 1, "summer's": 2, 'more': 3, 'and': 4, 'shall': 5, 'i': 6, 'compare': 7, 'thee': 8, 'to': 9, 'day': 10, 'thou': 11, 'art': 12, 'lovely': 13, 'temperate': 14, 'rough': 15, 'winds': 16, 'do': 17, 'shake': 18, 'the': 19, 'darling': 20, 'buds': 21, 'of': 22, 'may': 23, 'lease': 24, 'hath': 25, 'all': 26, 'too': 27, 'short': 28, 'date': 29}


In [6]:
total_words = len(tokenizer.word_index) + 1
total_words

30

In [7]:
input_sequence = []

for line in text.split('\n'):

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1,len(token_list)):

        n_gram_sequence = token_list[:i+1]
        input_sequence.append(n_gram_sequence)
input_sequence
    

[[5, 6],
 [5, 6, 7],
 [5, 6, 7, 8],
 [5, 6, 7, 8, 9],
 [5, 6, 7, 8, 9, 1],
 [5, 6, 7, 8, 9, 1, 2],
 [5, 6, 7, 8, 9, 1, 2, 10],
 [5, 6, 7, 8, 9, 1, 2, 10, 11],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3, 13],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3, 13, 4],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3, 13, 4, 3],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3, 13, 4, 3, 14],
 [15, 16],
 [15, 16, 17],
 [15, 16, 17, 18],
 [15, 16, 17, 18, 19],
 [15, 16, 17, 18, 19, 20],
 [15, 16, 17, 18, 19, 20, 21],
 [15, 16, 17, 18, 19, 20, 21, 22],
 [15, 16, 17, 18, 19, 20, 21, 22, 23],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24, 25],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24, 25, 26],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24, 25, 26, 27],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24, 25, 26, 27, 28],
 [1

In [8]:
# dowloading the Glove 

import os
import zipfile
import urllib.request

glove_zip = "glove.6B.zip"
glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"

if not os.path.exists(glove_zip):
    print("Downloading GloVe...")
    urllib.request.urlretrieve(glove_url, glove_zip)
    print("Download completed.")
else:
    print("GloVe ZIP already exists.")

# Extract only the 100-dimensional GloVe file
glove_file = "glove.6B.100d.txt"

if not os.path.exists(glove_file):
    with zipfile.ZipFile(glove_zip, "r") as zip_ref:
        zip_ref.extract(glove_file)

print("GloVe file ready:", glove_file)

Download completed.
GloVe file ready: glove.6B.100d.txt


In [9]:
embedding_dim = 100

embeddings_index = {}

with open(glove_file, encoding="utf-8") as f:

    for line in f:

        values = line.split()

        word = values[0]

        coefs = np.asarray(
            values[1:],
            dtype="float32"
        )

        embeddings_index[word] = coefs

print("Number of GloVe words:", len(embeddings_index))

print(
    "Vector shape for 'king':",
    embeddings_index["king"].shape
)

Number of GloVe words: 400000
Vector shape for 'king': (100,)


In [14]:
len(values)
# so here one word and 100 values 
# thats why word = values[0]
# and coefficient that 100  is used 
# coef = np.asarray(values[1:],dtype ='float32')

101

In [15]:
# creating the Glove Embedding Matrix 

embedding_matrix = np.zeros(
    (total_words, embedding_dim)
)

for word, index in tokenizer.word_index.items():

    embedding_vector = embeddings_index.get(word)

    if embedding_vector is not None:

        embedding_matrix[index] = embedding_vector

print("Embedding matrix shape:")
print(embedding_matrix.shape)

Embedding matrix shape:
(30, 100)


In [17]:
word = "shall"

word_id = tokenizer.word_index[word]

print("Word:", word)
print("Word ID:", word_id)

print("\nGloVe Vector:")
print(embedding_matrix[word_id])

print("\nVector Shape:")
print(embedding_matrix[word_id].shape)

Word: shall
Word ID: 5

GloVe Vector:
[-0.10439     0.083304    0.13293999 -0.16866    -0.082145    0.65233999
 -0.65973997  1.10350001  0.16146    -0.53351003  0.27911001  0.37257999
  1.07959998  0.21854    -0.64756    -1.01470006  0.73515999  0.93480003
 -1.53830004  0.26008999 -0.03608    -0.77754998 -0.32216999 -0.52026999
 -0.29111001  0.29710999 -0.07961    -0.035583    0.49700999 -0.75305998
 -0.068808    0.20190001  0.066844   -0.24338999  0.13402     0.27981001
  0.13369    -0.4041     -0.017153   -0.60680002 -0.48795     0.41452
  0.099233   -0.63211     0.082186   -0.71535999 -0.10198    -0.14579
 -0.58886999 -1.17589998 -0.033689    0.073611    0.35155001  0.93018001
  0.12324    -0.32122999  0.69988    -0.17414001  0.90671003 -0.023537
 -0.20203     0.34547001 -1.19519997 -0.45107001  1.02330005  0.39941001
 -0.046984    0.59947997 -0.19896001 -0.94849002  0.46928999 -0.60995001
  0.85470003 -0.087851    0.39241001 -0.083171   -0.97579998 -0.15913001
 -0.64168    -0.68085

In [20]:
max_sequence_len = max(
    [len(x) for x in input_sequence]
)

print("Maximum sequence length:", max_sequence_len)

input_sequence = np.array(
    pad_sequences(
        input_sequence,
        maxlen=max_sequence_len,
        padding="pre"
    )
)

print("Input sequences shape:", input_sequence.shape)

Maximum sequence length: 18
Input sequences shape: (31, 18)


In [23]:
X = input_sequence[:, :-1]

y = input_sequence[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (31, 17)
y shape: (31,)


In [24]:
y = tf.keras.utils.to_categorical(
    y,
    num_classes=total_words
)

print("y shape:", y.shape)

y shape: (31, 30)


In [25]:
with tf.device("/GPU:0"):

    model = Sequential([
        
        Embedding(
            input_dim=total_words,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            input_length=max_sequence_len - 1,
            trainable=False
        ),

        LSTM(64),

        Dense(
            total_words,
            activation="softmax"
        )
    ])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 17, 100)           3000      
                                                                 
 lstm (LSTM)                 (None, 64)                42240     
                                                                 
 dense (Dense)               (None, 30)                1950      
                                                                 
Total params: 47,190
Trainable params: 44,190
Non-trainable params: 3,000
_________________________________________________________________


In [26]:
history = model.fit(
    X,
    y,
    epochs=100,
    verbose=1
)

Epoch 1/100
1/1 [==============================] - 6s 6s/step - loss: 3.4089 - accuracy: 0.0645
Epoch 2/100
1/1 [==============================] - 0s 20ms/step - loss: 3.3610 - accuracy: 0.0968
Epoch 3/100
1/1 [==============================] - 0s 12ms/step - loss: 3.3155 - accuracy: 0.0645
Epoch 4/100
1/1 [==============================] - 0s 14ms/step - loss: 3.2719 - accuracy: 0.1613
Epoch 5/100
1/1 [==============================] - 0s 13ms/step - loss: 3.2300 - accuracy: 0.1613
Epoch 6/100
1/1 [==============================] - 0s 15ms/step - loss: 3.1895 - accuracy: 0.1290
Epoch 7/100
1/1 [==============================] - 0s 13ms/step - loss: 3.1500 - accuracy: 0.1613
Epoch 8/100
1/1 [==============================] - 0s 14ms/step - loss: 3.1111 - accuracy: 0.1935
Epoch 9/100
1/1 [==============================] - 0s 14ms/step - loss: 3.0725 - accuracy: 0.1935
Epoch 10/100
1/1 [==============================] - 0s 16ms/step - loss: 3.0338 - accuracy: 0.1935
Epoch 11/100
1/1 [===

In [27]:
def generate_text(seed_text, next_words, max_sequence_len):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences(
            [seed_text]
        )[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len - 1,
            padding="pre"
        )

        predicted = np.argmax(
            model.predict(
                token_list,
                verbose=0
            ),
            axis=1
        )[0]

        output_word = ""

        for word, index in tokenizer.word_index.items():

            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

In [28]:
seeds = [
    "Shall I compare",
    "Thou art more",
    "Rough winds do",
    "summer's lease"
]

for seed in seeds:

    generated = generate_text(
        seed,
        8,
        max_sequence_len
    )

    print("Seed:", seed)
    print("Generated:", generated)
    print("-" * 60)

Seed: Shall I compare
Generated: Shall I compare thee to a summer's day thou art more
------------------------------------------------------------
Seed: Thou art more
Generated: Thou art more lovely and more temperate temperate temperate temperate temperate
------------------------------------------------------------
Seed: Rough winds do
Generated: Rough winds do shake the darling buds of may and summer's
------------------------------------------------------------
Seed: summer's lease
Generated: summer's lease i compare thee to a summer's day thou
------------------------------------------------------------
